_Copyright (C) 2024 Jagger Alexander_

_This program is free software: you can redistribute it and/or modify it as published._

_This program is distributed in the hope that it will be informative, but without any warranty; without even the implied warranty of merchantability or fitness for a particular purpose._

### Process SST data for ML-based heat wave prediction

1. Import necessary packages
2. Import downloaded SST data, combine, and output for prediction
3. Example visualization of SST regression

In [2]:
#1. Import necessary packages
#---

import xarray as xr
import pandas as pd
import glob
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats

In [3]:
#2. Import downloaded SST data
#---

# Define the directory containing the NetCDF files
directory = r"C:\Users\17753\Documents\AustinClim"
file_pattern = "sst.day.anom.*.nc"
file_paths = glob.glob(f"{directory}\\{file_pattern}")

# Define the latitude and longitude bounds for the Gulf of Mexico
lat_bounds = [20, 30]   # Latitude range for the Gulf of Mexico
lon_bounds = [82, 95]  # Longitude range for the Gulf of Mexico

# Initialize an empty list to store dataframes
dataframes = []

# Process each file
for file_path in sorted(file_paths):
    print(f"Processing {file_path}...")
    
    # Open the NetCDF file using xarray
    dataset = xr.open_dataset(file_path)
    
    # Access the 'anom' variable
    sst_anom = dataset['anom']
    
    # Subset the data to include only the Gulf of Mexico region
    sst_anom_gulf = sst_anom.sel(lat=slice(lat_bounds[0], lat_bounds[1]), lon=slice(lon_bounds[0], lon_bounds[1]))
    
    # Compute daily average SST anomaly for the Gulf of Mexico, ignoring NaNs
    daily_avg = sst_anom_gulf.mean(dim=['lat', 'lon'], skipna=True)
    
    # Convert to pandas DataFrame
    year = file_path.split('.')[-2]
    time_series_df = pd.DataFrame({
        'date': daily_avg.time.values,
        'anom': daily_avg.values,
        'year': year
    })
    
    # Append DataFrame to the list
    dataframes.append(time_series_df)
    
    # Close the dataset when done
    dataset.close()

# Concatenate all DataFrames vertically
combined_df = pd.concat(dataframes, ignore_index=True)

# Save the combined DataFrame to a CSV file
output_file = r"C:\Users\17753\Documents\AustinClim\gulf_sst_anom_daily_avg.csv"
combined_df.to_csv(output_file, index=False)

print("Daily average SST anomaly data saved to:", output_file)



Processing C:\Users\17753\Documents\AustinClim\sst.day.anom.2011.nc...
Processing C:\Users\17753\Documents\AustinClim\sst.day.anom.2012.nc...
Processing C:\Users\17753\Documents\AustinClim\sst.day.anom.2013.nc...
Processing C:\Users\17753\Documents\AustinClim\sst.day.anom.2014.nc...
Processing C:\Users\17753\Documents\AustinClim\sst.day.anom.2015.nc...
Processing C:\Users\17753\Documents\AustinClim\sst.day.anom.2016.nc...
Processing C:\Users\17753\Documents\AustinClim\sst.day.anom.2017.nc...
Processing C:\Users\17753\Documents\AustinClim\sst.day.anom.2018.nc...
Processing C:\Users\17753\Documents\AustinClim\sst.day.anom.2019.nc...
Processing C:\Users\17753\Documents\AustinClim\sst.day.anom.2020.nc...
Processing C:\Users\17753\Documents\AustinClim\sst.day.anom.2021.nc...
Processing C:\Users\17753\Documents\AustinClim\sst.day.anom.2022.nc...
Processing C:\Users\17753\Documents\AustinClim\sst.day.anom.2023.nc...
Daily average SST anomaly data saved to: C:\Users\17753\Documents\AustinClim\

In [4]:
#3. Example visualization of SST regression
#---

# Load the combined DataFrame from the CSV file
combined_df = pd.read_csv(r"C:\Users\17753\Documents\AustinClim\gulf_sst_anom_daily_avg.csv")

# Convert 'date' column to datetime format
combined_df['DATE'] = pd.to_datetime(combined_df['date'])

# Extract the year from the date
combined_df['year'] = combined_df['date'].dt.year

# Fit a linear regression model on the year
slope, intercept, r_value, p_value, std_err = stats.linregress(combined_df['year'], combined_df['anom'])

# Calculate the R-squared value
r_squared = r_value**2

# Generate the regression line
regression_line = intercept + slope * combined_df['year']

# Plot the 'anom' values as a time series
plt.figure(figsize=(12, 6))
plt.plot(combined_df['date'], combined_df['anom'], label='SST Anomaly', color='b')

# Plot the regression line
plt.plot(combined_df['date'], regression_line, color='r', linestyle='--', label=f'Regression Line: $y = {slope:.2f}x + {intercept:.2f}$\n$R^2 = {r_squared:.2f}$')

# Format the x-axis to show yearly ticks
ax = plt.gca()
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.xlabel('Date')
plt.ylabel('SST Anomaly (°C)')
plt.title('Daily Sea Surface Temperature Anomaly (Gulf of Mexico)')
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()

# Show plot
plt.show()

# Print regression details
print(f"Regression Equation: y = {slope:.2f}x + {intercept:.2f}")
print(f"R-squared: {r_squared:.2f}")

AttributeError: Can only use .dt accessor with datetimelike values